In [1]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

try:
    import keras_tuner as kt

    HAS_KERAS_TUNER = True
except Exception:
    kt = None
    HAS_KERAS_TUNER = False

print("TensorFlow:", tf.__version__)
print("Keras Tuner available:", HAS_KERAS_TUNER)
if HAS_KERAS_TUNER:
    print("Keras Tuner:", kt.__version__)

IMG_W = int(os.environ.get("IMG_W", "128"))
IMG_H = int(os.environ.get("IMG_H", "128"))
BATCH = int(os.environ.get("BATCH", "16"))
SEED = 123
AUTOTUNE = tf.data.AUTOTUNE

MAX_EPOCHS = int(os.environ.get("TUNER_MAX_EPOCHS", "15"))
PROJECT_NAME = os.environ.get("TUNER_PROJECT", "kt_chest_xray")

MODEL_KIND = os.environ.get("MODEL_KIND", "improved").strip().lower()
FIT = os.environ.get("FIT", "0").strip() not in {"0", "false", "False"}
SHOW_PLOTS = os.environ.get("SHOW_PLOTS", "1").strip() not in {"0", "false", "False"}
RUN_GRADCAM = os.environ.get("RUN_GRADCAM", "0").strip() not in {"0", "false", "False"}
RUN_THRESHOLD = os.environ.get("RUN_THRESHOLD", "0").strip() not in {"0", "false", "False"}
SAVE_EXTRAS = os.environ.get("SAVE_EXTRAS", "0").strip() not in {"0", "false", "False"}
BINARY_PNEUMONIA = os.environ.get("BINARY_PNEUMONIA", "1").strip() not in {"0", "false", "False"}

EPOCHS_BASELINE = int(os.environ.get("EPOCHS_BASELINE", "8"))
EPOCHS_IMPROVED = int(os.environ.get("EPOCHS_IMPROVED", "25"))


TensorFlow: 2.18.1
Keras Tuner available: True
Keras Tuner: 1.4.8


In [2]:
def resolve_data_root() -> Path:
    env_root = os.environ.get('CHEST_XRAY_ROOT', '').strip()
    if env_root:
        return Path(env_root)

    project_root = Path('.').resolve()
    if (project_root / 'train').is_dir() and (project_root / 'test').is_dir():
        return project_root

    chest_xray = project_root / 'chest_xray'
    if (chest_xray / 'train').is_dir() and (chest_xray / 'test').is_dir():
        return chest_xray

    raise SystemExit(
        'Dataset not found. Set CHEST_XRAY_ROOT to the folder containing train/ and test/.'
    )


def count_images(split_dir: Path) -> dict[str, int]:
    counts: dict[str, int] = {}
    for cls in sorted(p.name for p in split_dir.iterdir() if p.is_dir()):
        d = split_dir / cls
        n = 0
        for pat in ('*.jpeg', '*.jpg', '*.png'):
            n += sum(1 for _ in d.glob(pat))
        counts[cls] = n
    return counts


data_root = resolve_data_root()
train_dir = data_root / 'train'
test_dir = data_root / 'test'

print('Data root:', data_root)
print('Train counts:', count_images(train_dir))
print('Test counts:', count_images(test_dir))

train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    seed=SEED,
    validation_split=0.2,
    subset='both',
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH,
    label_mode='int',
    shuffle=True,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=None,
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH,
    label_mode='int',
    shuffle=False,
)

class_names = list(train_ds.class_names)
num_classes = len(class_names)
print('Classes:', class_names)

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


Data root: C:\Users\leona\OneDrive\Uni\Year 4\Semester 2\Comp Vision\Ca2
Train counts: {'BACTERIAL': 2596, 'NORMAL': 1461, 'VIRAL': 1362}
Test counts: {'BACTERIAL': 184, 'NORMAL': 122, 'VIRAL': 131}
Found 5419 files belonging to 3 classes.
Using 4336 files for training.
Using 1083 files for validation.
Found 437 files belonging to 3 classes.
Classes: ['BACTERIAL', 'NORMAL', 'VIRAL']


In [3]:
# Class weights for imbalance
train_counts = count_images(train_dir)
y_int = np.concatenate(
    [np.full(train_counts[c], i, dtype=np.int32) for i, c in enumerate(class_names)]
)
cw = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_int)
class_weight = {i: float(cw[i]) for i in range(num_classes)}
print('class_weight:', class_weight)


class_weight: {0: 0.695814072932717, 1: 1.2363677846224048, 2: 1.3262359275575135}


In [4]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomContrast(0.1),
    ],
    name='aug',
)


def build_tunable_model(hp) -> tf.keras.Model:
    if not HAS_KERAS_TUNER:
        raise SystemExit(
            "keras-tuner is not installed. Install it (pip install keras-tuner) or skip the tuner section."
        )

    lr = hp.Float('lr', min_value=1e-4, max_value=3e-3, sampling='log')
    dropout = hp.Float('dropout', min_value=0.2, max_value=0.6, step=0.05)
    dense_units = hp.Choice('dense_units', values=[64, 128, 256])

    f1 = hp.Choice('filters1', values=[16, 32])
    f2 = hp.Choice('filters2', values=[32, 64])
    f3 = hp.Choice('filters3', values=[64, 128])

    inputs = tf.keras.layers.Input(shape=(IMG_H, IMG_W, 3))
    x = tf.keras.layers.Rescaling(1.0 / 255)(inputs)
    x = data_augmentation(x)

    x = tf.keras.layers.Conv2D(f1, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(f2, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(f3, 3, padding='same', activation='relu', name='last_conv')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(dense_units, activation='relu')(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name='tuned_cnn')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    return model


In [5]:
if not HAS_KERAS_TUNER:
    print(
        "Skipping Keras Tuner section (keras-tuner not installed)."
        "\nInstall with: pip install keras-tuner"
    )
else:
    tuner = kt.Hyperband(
        build_tunable_model,
        objective="val_accuracy",
        max_epochs=MAX_EPOCHS,
        factor=3,
        directory="kt_dir",
        project_name=PROJECT_NAME,
        overwrite=True,
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        ),
    ]

    t0 = time.perf_counter()
    tuner.search(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )
    print("Tuning wall time (s):", round(time.perf_counter() - t0, 1))

    best_hp = tuner.get_best_hyperparameters(1)[0]
    print("\nBest hyperparameters:")
    for k in sorted(best_hp.values.keys()):
        print(f"- {k}: {best_hp.get(k)}")


UnknownError: Failed to remove a directory: \\?\c:\Users\leona\OneDrive\Uni\Year 4\Semester 2\Comp Vision\Ca2\kt_dir\kt_chest_xray/trial_0013; Input/output error

In [ ]:
if not HAS_KERAS_TUNER:
    print("Skipping tuned-model train/eval (keras-tuner not installed).")
else:
    best_model = tuner.hypermodel.build(best_hp)

    history = best_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=4, restore_best_weights=True
            )
        ],
        class_weight=class_weight,
        verbose=1,
    )

    test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
    print(f"Test loss={test_loss:.4f}  Test accuracy={test_acc:.4f}")

    proba = best_model.predict(test_ds, verbose=1)
    y_true = np.concatenate([lb.numpy() for _, lb in test_ds], axis=0)
    y_pred = np.argmax(proba, axis=1)

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0,
        )
    )

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm)
    ax.set_xticks(range(num_classes), class_names, rotation=45, ha="right")
    ax.set_yticks(range(num_classes), class_names)
    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                color="w" if cm[i, j] > cm.max() / 2 else "k",
            )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion matrix (tuned model)")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    fig.savefig("confusion_matrix_tuned.png", dpi=150, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close("all")

    fig = plt.figure()
    plt.plot(history.history.get("accuracy", []), label="train acc")
    plt.plot(history.history.get("val_accuracy", []), label="val acc")
    plt.plot(history.history.get("loss", []), label="train loss")
    plt.plot(history.history.get("val_loss", []), label="val loss")
    plt.xlabel("epoch")
    plt.legend()
    plt.title("Tuned model — train vs val")
    plt.tight_layout()
    fig.savefig("curves_tuned.png", dpi=150, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close("all")

    best_model.save("best_tuned.keras")
    print("Saved best model to best_tuned.keras")


In [ ]:
def finish_figure(fig: plt.Figure | None = None, *, filename: str | None = None) -> None:
    """Show or save a Matplotlib figure depending on SHOW_PLOTS / SAVE_EXTRAS."""
    if SHOW_PLOTS:
        plt.show()
        return

    if fig is not None and filename:
        out_path = Path.cwd() / filename
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close("all")

print("\nReusing already-loaded datasets for the full pipeline...")

try:
    normal_idx = next(i for i, n in enumerate(list(class_names)) if n.upper() == "NORMAL")
except StopIteration:
    normal_idx = None

if BINARY_PNEUMONIA:
    if normal_idx is None:
        raise SystemExit("BINARY_PNEUMONIA=1 requires a NORMAL class folder to be present.")

    def to_binary(ds: tf.data.Dataset) -> tf.data.Dataset:
        return ds.map(
            lambda x, y: (x, tf.cast(tf.not_equal(y, normal_idx), tf.int32)),
            num_parallel_calls=AUTOTUNE,
        )

    train_ds = to_binary(train_ds)
    val_ds = to_binary(val_ds)
    test_ds = to_binary(test_ds)
    class_names = ["NORMAL", "PNEUMONIA"]
    num_classes = 2
    print("Using binary labels: NORMAL vs PNEUMONIA (not NORMAL)")

train_counts = count_images(train_dir)

if sum(train_counts.values()) > 0:
    if BINARY_PNEUMONIA:
        normal_n = int(train_counts.get("NORMAL", 0))
        sick_n = int(sum(v for k, v in train_counts.items() if k.upper() != "NORMAL"))
        y_int = np.concatenate(
            [
                np.zeros(normal_n, dtype=np.int32),
                np.ones(sick_n, dtype=np.int32),
            ]
        )
        cw = compute_class_weight(
            class_weight="balanced", classes=np.array([0, 1]), y=y_int
        )
        class_weight_dict = {0: float(cw[0]), 1: float(cw[1])}
    else:
        y_int = np.concatenate(
            [np.full(train_counts[c], i, dtype=np.int32) for i, c in enumerate(class_names)]
        )
        cw = compute_class_weight(
            class_weight="balanced", classes=np.arange(num_classes), y=y_int
        )
        class_weight_dict = {i: float(cw[i]) for i in range(num_classes)}
else:
    class_weight_dict = None

print("class_weight (pipeline):", class_weight_dict)

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

fig = plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(6, int(images.shape[0]))):
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i].numpy())])
        plt.axis("off")

if SAVE_EXTRAS:
    finish_figure(fig, filename="train_samples.png")
else:
    plt.close("all")


In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomContrast(0.1),
    ],
    name="aug",
)


def build_baseline(num_classes: int) -> tf.keras.Model:
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(IMG_H, IMG_W, 3)),
            tf.keras.layers.Rescaling(1.0 / 255),
            tf.keras.layers.Conv2D(16, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(32, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(32, 3, activation="relu", name="last_conv"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(512, activation="relu"),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(num_classes, activation="softmax"),
        ],
        name="baseline_cnn",
    )


def build_improved(num_classes: int) -> tf.keras.Model:
    inputs = tf.keras.layers.Input(shape=(IMG_H, IMG_W, 3))
    x = tf.keras.layers.Rescaling(1.0 / 255)(inputs)
    x = data_augmentation(x)
    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu", name="last_conv")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="improved_cnn")


def build_transfer(num_classes: int) -> tf.keras.Model:
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_H, IMG_W, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False
    inputs = tf.keras.layers.Input(shape=(IMG_H, IMG_W, 3))
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    m = tf.keras.Model(inputs, outputs, name="mobilenet_transfer")
    m._transfer_base = base
    return m


def make_model_and_checkpoint(kind: str, num_classes: int) -> tuple[tf.keras.Model, Path, int, list]:
    suffix = "_bin" if BINARY_PNEUMONIA else ""

    if kind == "baseline":
        model = build_baseline(num_classes)
        ckpt = Path.cwd() / f"best_baseline{suffix}.keras"
        epochs = EPOCHS_BASELINE
        callbacks: list[tf.keras.callbacks.Callback] = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=str(ckpt),
                monitor="val_accuracy",
                mode="max",
                save_best_only=True,
                verbose=1,
            )
        ]
        return model, ckpt, epochs, callbacks

    if kind == "transfer":
        model = build_transfer(num_classes)
        ckpt = Path.cwd() / f"best_transfer{suffix}.keras"
        epochs = 6
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
            ),
            tf.keras.callbacks.ModelCheckpoint(
                filepath=str(ckpt),
                monitor="val_accuracy",
                mode="max",
                save_best_only=True,
                verbose=1,
            ),
        ]
        return model, ckpt, epochs, callbacks

    # default improved
    model = build_improved(num_classes)
    ckpt = Path.cwd() / f"best_improved{suffix}.keras"
    epochs = EPOCHS_IMPROVED
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
    ]
    return model, ckpt, epochs, callbacks


In [ ]:
model, checkpoint_path, epochs, callbacks = make_model_and_checkpoint(MODEL_KIND, num_classes)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

if not FIT and not checkpoint_path.exists():
    print(
        "FIT=0 but checkpoint missing; running a short 1-epoch fit to create it:",
        checkpoint_path,
    )
    FIT = True
    epochs = 1

if FIT:
    t0 = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1,
    )
    train_s = time.perf_counter() - t0
    print(f"\nTrain wall time ({MODEL_KIND}): {train_s:.1f} s")

    if MODEL_KIND == "transfer":
        base = getattr(model, "_transfer_base", None)
        if base is not None:
            base.trainable = True
            for layer in base.layers[:-30]:
                layer.trainable = False
            model.compile(
                optimizer=tf.keras.optimizers.Adam(1e-5),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                metrics=["accuracy"],
            )
            t1 = time.perf_counter()
            model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=4,
                class_weight=class_weight_dict,
                verbose=1,
            )
            ft_s = time.perf_counter() - t1
            print(f"Fine-tune wall time ({MODEL_KIND}): {ft_s:.1f} s")
else:
    model = tf.keras.models.load_model(str(checkpoint_path))

if checkpoint_path.exists():
    model = tf.keras.models.load_model(str(checkpoint_path))

out_units = int(model.output_shape[-1])
if out_units != num_classes:
    raise SystemExit(
        f"Loaded model has {out_units} outputs but current task expects {num_classes}. "
        f"Set FIT=1 to train a new checkpoint at {checkpoint_path} (or disable BINARY_PNEUMONIA)."
    )

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test loss={test_loss:.4f}  Test accuracy={test_acc:.4f}")

y_true = np.concatenate([lb.numpy() for _, lb in test_ds], axis=0)
proba = model.predict(test_ds, verbose=1)
y_pred = np.argmax(proba, axis=1)

print("\nClassification report:")
print(
    classification_report(
        y_true, y_pred, target_names=list(class_names), digits=4, zero_division=0
    )
)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_xticks(range(num_classes), list(class_names), rotation=45, ha="right")
ax.set_yticks(range(num_classes), list(class_names))
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color="w" if cm[i, j] > cm.max() / 2 else "k",
        )
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (test)")
fig.colorbar(im, ax=ax)
plt.tight_layout()
finish_figure(fig, filename=f"confusion_matrix_{MODEL_KIND}.png")

if FIT and "history" in locals():
    fig = plt.figure()
    plt.plot(history.history.get("accuracy", []), label="train acc")
    plt.plot(history.history.get("val_accuracy", []), label="val acc")
    plt.plot(history.history.get("loss", []), label="train loss")
    plt.plot(history.history.get("val_loss", []), label="val loss")
    plt.xlabel("epoch")
    plt.legend()
    plt.title(f"{MODEL_KIND} — train vs val")
    plt.tight_layout()
    finish_figure(fig, filename=f"curves_{MODEL_KIND}.png")


In [ ]:
def grad_cam(m: tf.keras.Model, img: tf.Tensor, *, layer_name: str) -> np.ndarray:
    img = tf.cast(img, tf.float32)
    if img.shape.rank == 3:
        img = tf.expand_dims(img, 0)

    grad_model = tf.keras.Model(m.inputs, [m.get_layer(layer_name).output, m.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img, training=False)
        pred_idx = tf.argmax(preds[0])
        class_channel = preds[:, pred_idx]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(1, 2))

    conv_out = conv_out[0]
    weights = pooled[0]
    heatmap = tf.reduce_sum(conv_out * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    denom = tf.reduce_max(heatmap) + 1e-8
    heatmap = heatmap / denom
    return heatmap.numpy()


def overlay_heatmap(img_uint8: np.ndarray, heatmap: np.ndarray) -> np.ndarray:
    hm = tf.convert_to_tensor(heatmap[..., np.newaxis], dtype=tf.float32)
    hm = tf.image.resize(hm, (IMG_H, IMG_W), method="bilinear")[..., 0].numpy()

    cmap = plt.get_cmap("jet")
    colored = cmap(hm)[..., :3]
    colored = (255 * colored).astype(np.uint8)
    overlay = (0.6 * img_uint8 + 0.4 * colored).clip(0, 255).astype(np.uint8)
    return overlay


if RUN_GRADCAM:
    layer_name = "last_conv"
    fig = plt.figure(figsize=(10, 6))
    k = 0
    for images, labels in test_ds.take(1):
        for i in range(min(3, int(images.shape[0]))):
            img_u8 = images[i].numpy().astype("uint8")
            hm = grad_cam(model, images[i], layer_name=layer_name)
            ov = overlay_heatmap(img_u8, hm)

            plt.subplot(2, 3, k * 2 + 1)
            plt.imshow(img_u8)
            plt.title(f"Original (true={class_names[int(labels[i])]})")
            plt.axis("off")

            plt.subplot(2, 3, k * 2 + 2)
            plt.imshow(ov)
            plt.title("Grad-CAM overlay")
            plt.axis("off")
            k += 1

    plt.suptitle(f"Grad-CAM ({MODEL_KIND})")
    plt.tight_layout()
    finish_figure(fig, filename=f"gradcam_{MODEL_KIND}.png")


In [ ]:
if RUN_THRESHOLD:
    if num_classes == 2:
        print("\nThreshold sweep: P(PNEUMONIA) >= t")
        val_true = np.concatenate([lb.numpy() for _, lb in val_ds], axis=0).astype(np.int32)
        val_probs = model.predict(val_ds, verbose=1)
        val_p_sick = val_probs[:, 1]

        test_true = y_true.astype(np.int32)
        test_p_sick = proba[:, 1]
        pos_name = "PNEUMONIA"
    else:
        if normal_idx is None:
            print("Threshold sweep skipped (NORMAL class required).")
            val_p_sick = None
        else:
            print("\nThreshold sweep (screening): sick = not NORMAL, p(sick) = 1 - p(NORMAL)")
            val_true_3 = np.concatenate([lb.numpy() for _, lb in val_ds], axis=0).astype(np.int32)
            val_probs = model.predict(val_ds, verbose=1)
            val_p_sick = 1.0 - val_probs[:, normal_idx]
            val_true = (val_true_3 != int(normal_idx)).astype(np.int32)

            test_true = (y_true.astype(np.int32) != int(normal_idx)).astype(np.int32)
            test_p_sick = 1.0 - proba[:, normal_idx]
            pos_name = "SICK(not NORMAL)"

    if val_p_sick is not None:
        best_t, best_f1 = 0.5, -1.0
        for t in np.linspace(0.05, 0.95, 19):
            pred_sick = (val_p_sick >= t).astype(np.int32)
            f1 = f1_score(val_true, pred_sick)
            if f1 > best_f1:
                best_f1, best_t = float(f1), float(t)

        print(f"Best val threshold (F1, pos={pos_name}): t={best_t:.3f}  F1={best_f1:.4f}")

        test_pred_sick = (test_p_sick >= best_t).astype(np.int32)
        tp = int(np.sum((test_true == 1) & (test_pred_sick == 1)))
        tn = int(np.sum((test_true == 0) & (test_pred_sick == 0)))
        fp = int(np.sum((test_true == 0) & (test_pred_sick == 1)))
        fn = int(np.sum((test_true == 1) & (test_pred_sick == 0)))
        prec = tp / (tp + fp + 1e-9)
        rec = tp / (tp + fn + 1e-9)
        f1 = 2 * prec * rec / (prec + rec + 1e-9)
        print(f"Test confusion: TP={tp}, FP={fp}, TN={tn}, FN={fn}")
        print(f"Test: precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")


In [ ]:
print("\nNOTEBOOK_DONE")
print("MODEL_KIND:", MODEL_KIND)
print("BINARY_PNEUMONIA:", BINARY_PNEUMONIA)
print("classes:", list(class_names))
try:
    print("checkpoint_path:", str(checkpoint_path))
except Exception:
    pass
try:
    print("test_accuracy:", float(test_acc))
except Exception:
    pass
